# GPU ant sweep: a CUDA batch runner for (rule, seed) ants

## Question and why it matters
Can one RTX 4000 Ada run thousands of independent ants at once, returning exactly what the CPU oracle chain (`reference -> fast -> lean`) returns, and how much faster is it than `sweep_lean` on 6 cores? Census-style sweeps are the bottleneck of the project's larger questions. Links: `README.md`, `NOTEBOOK.md`, code in `src/gpu/`, `src/bin/gpu_sweep.rs`, tests in `tests/gpu.rs`, `tests/onset.rs`.

## Assumptions and definitions
- A job's *seed* is the random-background seed of `ext::bg_colour`; `None` is the blank census background (the ant itself is deterministic). Not turn noise.
- At most 128 colours. A job larger than the whole tile pool is an error (never a wrong row).
- Speedups are whole-process wall time against `sweep_lean` on 6 rayon threads, one run per cell (variance not measured).


## Provenance
Prints the git commit, date and a SHA-256 for every input file. Inputs are the committed benchmark logs and their summary.

In [ ]:
import hashlib, json, subprocess, datetime
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
EXP = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "NOTEBOOK.md").exists())
INPUTS = [EXP / "results" / f for f in ["bench_L2-12-1e6.log", "bench_L2-12-1e8.log", "bench_summary.json"]]
assert all(p.exists() for p in INPUTS), [p for p in INPUTS if not p.exists()]
sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
print("git", sha, "| run", datetime.datetime.now().isoformat(timespec="seconds"), "| no randomness in this notebook")
for p in INPUTS:
    print(hashlib.sha256(p.read_bytes()).hexdigest()[:16], p.relative_to(ROOT))


## Data and method
`scripts/bench.sh MIN MAX STEPS TAG "LANES..."` runs `sweep_lean` once and `gpu_sweep` at each lane count on the same census slice (lengths 2..=12, 4083 rules, grid 8192) and `cmp`s the CSVs; `scripts/summarise_bench.py` turns the logs into `bench_summary.json`. The heavy computation is `src/gpu/ant.cu` (one thread per ant, a virtual 8192x8192 grid as 64x64 tiles from a global pool, headings logged 2 bits/step) plus unchanged CPU highway detection.

In [ ]:
summary = json.loads((EXP / "results" / "bench_summary.json").read_text())
for tag, d in summary.items():
    print(f"{tag}: sweep_lean {d['lean_wall_s']:.1f} s")
    for r in d["gpu"]:
        print(f"  {r['lanes']:>5} lanes  {r['wall_s']:6.1f} s  {r['speedup_vs_lean']:5.2f}x  device path {r['device_path_steps_per_s']:.2e} steps/s  CSV {r['csv']}")


## Results
**Correctness.** Every GPU CSV compared is byte-identical to `sweep_lean`'s, and the proptests agree with `fast::run` on every `RunResult` field (grid of exactly one tile, ragged grid, forced pool starvation and requeue, many lanes) and with a dense CPU reference for seeded backgrounds. It does not show agreement for rules with more than 12 colours beyond the unit tests, nor for step budgets above 1e8.

**Speed.** At 1e8 steps the GPU is 4.9x faster than `sweep_lean` at 4096 lanes and its throughput was still rising linearly with lanes. At 1e6 steps it is 11.8x, but there `sweep_lean` is dominated by per-rule overhead, so that figure is not a stepping comparison. It does not show saturation (only 4083 jobs), variance, or behaviour with many more jobs than lanes.


## Caveats and controls
- Contamination: a first 1e6 run and the first 1e8 lean run overlapped another experiment's probe on all cores; those numbers were discarded and re-measured on an idle machine (see `README.md`).
- Registered predictions: P1 (identical output) held; P2 (3-8x, fail if <2x) held at 1e8, exceeded at 1e6 for the reason above; P3 (saturation at 2k-8k lanes) was not testable with 4083 jobs.
- Highway onset replay (CPU) is 9 s of 59 s at 1e8 steps.

## Next steps
Port the probe statistics (visited set, perimeter, gyration, octave windows) so `probe 27 8192` batches can use the GPU; a GPU onset pass; cache the NVRTC PTX.